# Lab - YOLOv4-Tiny Part 2 - Darknet Setup
## E6792 Spring 2026


In this Lab you will train YOLOv4-Tiny using the Darknet framework. Darknet is an open source neural network framework written in C and CUDA, which means it's efficient and outperforms TensorFlow/PyTorch in terms of training times for similar networks. However, it may feel slightly less intuitive to a Python programmer. This notebook will guide you through the setup process. 

**You should execute these cells on the GCP instance.**

The first step in setting up Darknet is to specify the build configuration. This is done by editing the Makefile. We need to specify `GPU=1`, `CUDNN=1`, `CUDNN_HALF=1`, and `OPENCV=1` to use the GPU for parallel operations, the cuDNN GPU acceleration libraries, half precision operations, and OpenCV for loading images. We also need to specify the GPU architecture by uncommenting the `ARCH` variable that corresponds to your instance's GPU. 

<font color="red"><strong>TODO:</strong></font>  Open **darknet/Makefile** and make the changes specified above. 

## Install OpenCV

We need to install the C distribution of OpenCV (Open Source Computer Vision Library) for image processing. 

<font color="red"><strong>TODO:</strong></font>  Open a terminal and enter the command `sudo apt-get install libopencv-dev` to install OpenCV.

## Build Darknet

To compile the darknet project code, we simply execute the command `make` in the darknet directory. The Makefile specifies the configuration details of the Darknet setup. All dependancies are all located within the directory. 

<font color="red"><strong>TODO:</strong></font> In the terminal, navigate to the `darknet` directory and enter `make`. You will see the compilation output and several warnings, which is OK. The darknet code should execute without errors. 

After you've compiled the Darknet code and completed the discussion questions below, you can start on the training notebook **darknet/DarknetTraining.ipynb**.

## Discussion Questions

The Darknet framework defines CNN layers in CUDA such that they can be calculated on GPUs - but it also has a lot more functionality. For instance, there are libraries of different activation functions, layer types, and loss functions all implemented in CUDA and C that can be combined to generate complex model architectures. To get a better sense of how Darknet is combining CPU and GPU functionality to complete the computations necessary for Deep Learning, take a look at **darknet/src**. This directory contains the Darknet source code for individual model operations.

### Question 1
Open **darknet/src/activation_kernels.cu**, **darknet/src/activation_layer.c**, and **darknet/src/activation_layer.h** and look through the functions in these files. 

Describe the purpose of these functions and their designation to **.cu**, **.c**, or **.h**. How do these C and CUDA functions work to produce activation functions? How do they calculate the gradients of these activation functions?

<font color="red"><strong>TODO:</strong></font> 
#### **The purpose of functions:** ####
##### **darknet/src/activation_layer.c and **darknet/src/activation_layer.h**:** #####

Thes functions define the activation layer in Darknet and integrate it into the network execution pipeline. 


To be more specific, they:


1. create and initialize activation layers

2. run the forward by applying an activation function

3. run the backward by multiplying upstream gradients by calaculating activation derivative







##### **darknet/src/activation_kernels.cu** #####

This file implements the GPU side of activations and their gradients. 



It includes:

1. many device activation formulas and derivative formulas for calculation
2. helpers functions such as activate_kernel nnd gradient_kernel that helps to choose the formula based on a enum be named ACTIVATION
3. global CUDA kernels that apply activations and gradient calaulation over arrays that works as parallel programming,
4. extern "C" wrapper functions to launch the right kernel efficiently




#### **Their designation to .cu, .c, or .h** ####
.h: header： declares functions, types, and GPU functions if enabled; enables cross-file compilation and linking.




.c: CPU-side layer implementation: constructs the layers used for training, manage memory, and defines the training logic such as how to forward and how to backword. Besides, it also provides GPU wrapper functions when compiled with GPU (by defining #ifdef GPU blocks).






.cu: CUDA implementation: The functions in activation_kernels.cu are mostly defined for CUDA to execute huge amount of parallel activation and gradient computations on GPU.
IT also export bu defining '''extern "C"''' for C code to launch.




#### **How do these C and CUDA functions work to produce activation functions?** ####
During the forward propagation process, activation_layer.c will copy the state.input to the output buffer of the corresponding layer, and then it will apply activation function on the CPU. This is accomplished by the function activate_array(). 




On the GPU, a function called activate_array_ongpu() will start a CUDA kernel and perform parallel computations using CUDA to process each element separately. The type of the activation function is determined by the information stored in layer.activation, and the CUDA path is specified by the device function called through activate_kernel().


#### **How do they calculate the gradients of these activation functions?** ####
On CPU, the gradient_array(output, n, activation, delta) functio computes the derivative of the activation functionand scales the gradient.


On GPU, gradient_array_ongpu launches CUDA kernels such as gradient_array_kernel, which performs the operation:



Many derivatives are computed using the already-activated output value. SUch as sigmoid and tanh. Because of this, the gradient functions often take l.output or l.output_gpu as input.


For activation functions, darknet stores the pre-activation input values by functions like activation_input_gpu and uses specialized backward kernels such as gradient_array_mish_kernel to compute.




### Question 2

Open **darknet/src/dark_cuda.c**. Describe the purpose of the following functions: **cuda_free()**, **cuda_free_host()**, **cuda_push_array()**, **cuda_pull_array()**, **cuda_pull_array_async()**, and **get_number_of_blocks()**

<font color="red"><strong>TODO:</strong></font>  
**cuda_free()**
Frees GPU memory allocated by cudaMalloc





**cuda_free_host()**
Frees pinned host memory allocated by cudaHostAlloc




**cuda_push_array()**
Copy an array from CPU host memory to GPU memory (form host to gpu)





**cuda_pull_array()**
Copies an array from GPU memory back to CPU host memory (form gpu to host)




**cuda_pull_array_async()**
Starts an asynchronous copy from GPU to CPU but does not synchronize. This function is mainly used to execute data transfer with other work at the same time.




**get_number_of_blocks()**
Computes how many CUDA blocks are needed to cover array_size elements when each block has block_size threads.




What is the purpose of the "CHECK_CUDA(status);" lines in these functions?


<font color="red"><strong>TODO:</strong></font>  CHECK_CUDA(status) is Darknet’s error-checking macro for CUDA API calls. CUDA‘s functions usually return a status code like (cudaError_t). CHECK_CUDA verifies if the call  is succeeded. If not, it output error message and may stop execution to prevent failures that would perhaps cause incorrect results or confusing crashes later.

### Question 3

Briefly explain how the backward pass functions work (backward_convolutional_layer_gpu() and backward_maxpool_layer_kernel()).

<font color="red"><strong>TODO:</strong></font>  

`backward_convolutional_layer_gpu()` First, it applies the gradient of the activation function to `l.delta_gpu`, so the stored output gradients result are converted into gradients with respect to the pre-activation values. 

By the If statements, if batch normalization is used, it also backpropagates through batch normalization; otherwise it updates the bias gradients directly. Then it computes two main convolution backward results:  

1. **weight gradients**  It uses the current input and output gradients (from `cudnnConvolutionBackwardFilter` or GEMM,im2col in the non-cuDNN)

2. **input gradients**  This is for the previous layer using the filters and current layer gradients (from `cudnnConvolutionBackwardData` or GEMM,col2im).



In short, this function propagates error backward through activation, bias, batchnorm, and the convolution weights, and finally to the previous layer.




`backward_maxpool_layer_kernel()` backpropagates max pooling by sending each output gradient only to the input element that originally produced the maximum value in the pooling window. 


To be more specifuc, for each input position, the kernel checks nearby output positions which could have pooled from it, then it looks up the saved max index in `indexes`, and accumulates the corresponding `delta` if that input was the max-value. 

As the result, gradients flow only through the max locations, while non-max elements receive zero gradient.